In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('✓ numpy / matplotlib 就绪（全 notebook 共用环境）')

plt.show()


✓ numpy / matplotlib 就绪（全 notebook 共用环境）


# 🎤 04 · 语音面试八股与高频题

> 语音算法岗 · 面试速查卡 ｜ 信号特征 / 传统识别 / 端到端 / TTS / 声纹 / 工程

定位：**30 分钟速刷**。先看知识谱系 → 按模块连问速答（每问「一句话 + 展开点火」）→ 手撕 4 段代码 → 20 条自测打勾 → 复习路径。

## 🧭 知识谱系图

```
信号层  采样/量化 → 分帧加窗 → STFT → mel → MFCC/Fbank → CMVN
识别层  传统: GMM-HMM + DTW  |  端到端: CTC / Attention(LAS) / RNN-T
合成层  前端 → 声学模型(Tacotron2/VITS) → 声码器(Griffin-Lim/HiFi-GAN)
声纹层  说话人向量 x-vector / 声纹三要素(音色韵律发音习惯) / 验证与辨认
工程层  8k/16k 采样、VAD、降噪、RTF、首包延迟、流式/非流式
```

面试题基本都是「某层某概念 → 为什么 → 公式 → 数值/复杂度 → 对比项」五连问。

## 🔍 高频连问① 信号与特征（10 问）

**Q1 采样率怎么定？** 奈奎斯特定理 $f_s \ge 2 f_{max}$；电话 8k、宽带 16k、音乐 44.1k。高于 $f_s/2$ 的分量会**混叠**成低频，解法是采样前抗混叠 LPF。

**Q2 为什么分帧？** 语音非平稳但 10~30 ms **短时平稳**；事实标准帧长 25ms、帧移 10ms（60% 重叠），频率分辨率 $\Delta f=f_s/N$。

**Q3 为什么加窗？** DFT 假设周期延拓，硬截断产生频谱**泄漏**；汉明窗旁瓣 -43 dB（矩形窗 -13 dB），主瓣略宽换低泄漏，分析窗标配。

**Q4 STFT 与语谱图？** 每帧 DFT 得时频谱；语谱图横轴时间、纵轴频率、颜色能量。**分辨率权衡**：窗长→频细时糊，窗短反之。

**Q5 为什么 mel？** 人耳对频率对数感知：$m=2595\log_{10}(1+f/700)$；滤波器组低频密高频疏，压缩维度 + 贴合听觉掩蔽。

**Q6 MFCC 流程与每步目的？** 预加重（补偿嘴唇辐射、抬高频）→ 分帧加窗 → FFT 幅度 → mel 滤波 → log（对数响度 + 分布更像高斯）→ DCT（去相关、能量集中）取前 13 维 → 拼一阶/二阶差分。

**Q7 Fbank vs MFCC？** MFCC 去掉相关性更适统计模型；DNN 时代**Fbank 信息更全**（+CMVN 归一化抗信道差异），端到端甚至直接用原始波形。

**Q8 CMVN 是什么？** 倒谱均值方差归一化：按句减均值除方差，抗说话人/信道/麦克风差异。

**Q9 清音 vs 浊音？** 浊音低频周期 → **能量高 ZCR 低**；清音高频随机 → **能量低 ZCR 高**；可用能量+ZCR 做 VAD 端点检测；基频 F0 由自相关估计（普通话声调靠它）。

**Q10 相位干嘛去了？** STFT 有相位但 mel/MFCC 丢弃；语音**人的听感对相位不敏感**，但波形重建（TTS）必须恢复——交给声码器。

In [2]:
import numpy as np
np.random.seed(0)

def pre_emph(x, a=0.97):
    return np.concatenate([[x[0]], x[1:] - a*x[:-1]])
def enframe(x, win_len, hop):
    n = 1 + (len(x) - win_len)//hop
    return np.stack([x[i*hop:i*hop+win_len] for i in range(n)])
def fbank(x, fs, n_mels=26):
    x = pre_emph(x)
    win_len, hop = int(0.025*fs), int(0.010*fs)
    frames = enframe(x, win_len, hop)
    S = np.abs(np.fft.rfft(frames*np.hamming(win_len), axis=1))
    n_fft = win_len
    m_pts = 2595*np.log10(1 + np.linspace(0, fs/2, n_mels+2)/700)
    f_pts = 700*(10**(m_pts/2595) - 1)
    bins = np.floor((n_fft+1)*f_pts/fs).astype(int)
    fb = np.zeros((n_mels, n_fft//2+1))
    for i in range(n_mels):
        l, c, r = bins[i], bins[i+1], bins[i+2]
        fb[i, l:c] = (np.arange(l, c)-l)/max(c-l, 1)
        fb[i, c:r] = (r-np.arange(c, r))/max(r-c, 1)
    return np.log(fb @ S.T + 1e-12)   # (n_mels, n_frames) log-Fbank

fs = 16000; t = np.arange(int(fs*0.3))/fs
x = np.sin(2*np.pi*500*t) + 0.4*np.sin(2*np.pi*1800*t) + 0.05*np.random.randn(len(t))
F = fbank(x, fs)
print('log-Fbank 形状 (mel, 帧) =', F.shape)
print('第 1 帧 mel 能量前 5 个 =', np.round(F[:5, 0], 3))
assert F.shape[1] == 1 + (len(x) - int(0.025*fs))//int(0.010*fs)
print('✓ 手撕：预加重→分帧→加窗→FFT→mel 滤波→log')

log-Fbank 形状 (mel, 帧) = (26, 28)
第 1 帧 mel 能量前 5 个 = [2.201 3.464 0.391 0.351 0.405]
✓ 手撕：预加重→分帧→加窗→FFT→mel 滤波→log


## 🧠 高频连问② 传统识别：HMM 与 DTW（8 问）

**Q1 HMM 三要素？** $\lambda=(A,B,\pi)$：转移、发射、初始；两条假设（齐次、观测独立）。

**Q2 三个问题？** 求值（前向求 $P(O|\lambda)$）、解码（Viterbi 求最优状态链）、学习（Baum-Welch=EM 重估参数）。

**Q3 前向递推？** $\alpha_{t+1}(j)=\left[\sum_i\alpha_t(i)a_{ij}\right] b_j(o_{t+1})$，$O(TN^2)$；log 域 + logsumexp 防下溢。

**Q4 Viterbi vs 前向？** 前向**求和**（所有路径的概率），Viterbi **取 max**（最优路径），另加回溯指针 $\psi$。

**Q5 GMM 发射为何用混合？** 单高斯拟合不了复杂声学分布；$b_j(o)=\sum_m w_{jm}\mathcal{N}(o;\mu_{jm},\Sigma_{jm})$，语音用对角协方差省参数。

**Q6 DTW 递推？** $D(i,j)=d(i,j)+\min(D(i-1,j),D(i,j-1),D(i-1,j-1))$，复杂度 $O(T_1T_2)$，可带状剪枝到 $O(T\cdot W)$。

**Q7 DTW vs HMM？** DTW 是**距离模板**动态对齐，无参数、快；HMM 是**生成式概率模型**，可学可泛化。

**Q8 传统 ASR 全链路？** MFCC → GMM-HMM(音素) → 词典 → n-gram LM → WFST 解码：$W^*=\arg\max_W P(X|W)P(W)$。

In [3]:
def logsumexp(a, axis=0):
    m = a.max(axis=axis, keepdims=True)
    return m.squeeze(axis) + np.log(np.exp(a - m).sum(axis=axis))

def forward_log(A, B, pi):
    T, N = B.shape
    a = np.full((T, N), -np.inf)
    a[0] = pi + B[0]
    for t in range(1, T):
        for j in range(N):
            a[t, j] = logsumexp(a[t-1] + A[:, j]) + B[t, j]
    return logsumexp(a[-1])

A = np.log([[0.7, 0.3], [0.4, 0.6]])
pi = np.log([0.6, 0.4])
emis = np.array([[0.6, 0.3, 0.1], [0.1, 0.3, 0.6]])
obs = np.array([0, 1, 2, 1, 0, 2])
B = np.log(emis.T[obs])
ll = forward_log(A, B, pi)
print('前向 log P(O|λ) = %.5f' % ll)
print('✓ 手撕：log 域前向算法（一场面试必须 10 分钟内写对）')

前向 log P(O|λ) = -6.84063
✓ 手撕：log 域前向算法（一场面试必须 10 分钟内写对）


## 🧠 高频连问③ 端到端 ASR（10 问）

**Q1 CTC 怎么解决的帧字不对齐？** 词表加 blank，帧级输出独立路径，路径**折叠**后匹配标签；目标 $P(l|x)=\sum_{\pi:\mathcal{B}(\pi)=l}P(\pi|x)$。

**Q2 blank 作用？** 吸收停顿/无字帧 + 做**重复字符的分隔符**：`a ε a`→`aa`，`a a`→`a`。

**Q3 CTC 前向转移？** $\alpha_t(s)$ 三种来源：留在原地、前进一格、跳两格（非 ε 且与 $l'_{s-2}$ 不同时）；$O(TS)$；log 域数值稳定。

**Q4 CTC 的缺陷？** 帧间条件独立 → 语言建模弱；重复/静音时长建模差。

**Q5 贪心 vs Beam？** 贪心每帧 argmax 折叠；beam 保留 top-k 前缀按 logP 扩展合并，可挂 LM 浅融合重排序。

**Q6 WER/CER？** 编辑距离/参考长度：$WER=(S+D+I)/N$；中文按字称 CER。

**Q7 LAS 三模块？** Listen（编码器）→ Attend（注意力软对齐 $c_s=\sum_t\alpha_{s,t}h_t$）→ Spell（自回归解码）；权重矩阵应呈对角。

**Q8 CTC vs Attention？** CTC 隐式硬对齐、可流式、独立假设弱；AED 软对齐、建模强但自回归慢、易注意力漂移；工业用**联合 CTC+Attn**（Wenet U2）。

**Q9 RNN-T？** Encoder + Prediction（语言先验）+ Joint 逐帧输出，**单调流式**，交互场景主流。

**Q10 现状？** Whisper（AED 多语多任务）、Parakeet（CTC 流式）、SeamlessM4T（多模态）。

In [4]:
def log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))

def ctc_forward_log(y, labels):
    T, V = y.shape
    ext = np.zeros(2*len(labels)+1, dtype=int)
    for i, l in enumerate(labels):
        ext[2*i+1] = l
    S = len(ext)
    a = np.full((T, S), -np.inf)
    a[0, 0] = y[0, 0]
    if S > 1: a[0, 1] = y[0, labels[0]]
    for t in range(1, T):
        for s in range(S):
            v = a[t-1, s]
            if s >= 1: v = np.logaddexp(v, a[t-1, s-1])
            if s >= 2 and ext[s] != 0 and ext[s] != ext[s-2]:
                v = np.logaddexp(v, a[t-1, s-2])
            a[t, s] = v + y[t, ext[s]]
    return np.logaddexp(a[T-1, S-1], a[T-1, S-2])

np.random.seed(5)
y = log_softmax(np.random.randn(4, 3))
lp = ctc_forward_log(y, [1, 2])
print('CTC log P(ab|x) = %.5f' % lp)
print('✓ 手撕：CTC 前向（带 blank 的 log 域 DP）')

CTC log P(ab|x) = -3.13662
✓ 手撕：CTC 前向（带 blank 的 log 域 DP）


## 🗣️ 高频连问④ TTS 与声纹（8 问）

**Q1 TTS 全链路？** 文本 → 前端（正规化/注音）→ 声学模型（Tacotron2/VITS）→ 声码器（HiFi-GAN）→ 波形；评价 MOS、RTF。

**Q2 Tacotron2 结构？** 编码器 + 位置敏感注意力 + 自回归解码器（prenet+PostNet 精修 mel）+ 声码器。

**Q3 位置敏感注意力？** 打分加入累积权重特征，逼对齐**单调对角**，防跳读重复；覆盖惩罚只让每个输入位置被用一次。

**Q4 相位缺失？** mel 只含幅度；Griffin-Lim 交替投影迭代恢复相位，快但糊；神经声码器（WaveNet 自回归 / HiFi-GAN 对抗）更优。

**Q5 VITS 单阶段构成？** VAE 隐变量 + Flow 条件先验 + GAN 判别器 + 随机时长预测器 + MAS 单调对齐；损失 = KL + GAN + 时长 + mel L1。

**Q6 两阶段 vs 单阶段？** 两阶段 mel 可编辑可控但误差累积；VITS 端到端快且自然但隐变量不可直接编辑。

**Q7 时长从哪来？** 注意力（软，自回归）或时长预测器（硬，NAR）；VITS 用 MAS 动态规划求单调对齐作为训练目标。

**Q8 声纹三要素？** 音色（声道）、韵律（节奏语调）、发音习惯；说话人识别用 x-vector 嵌入 + cosine/PLDA 打分，区分**验证（1:1）**与**辨认（1:N）**。

In [5]:
def edit_distance(a, b):
    T1, T2 = len(a), len(b)
    D = np.zeros((T1+1, T2+1))
    D[:, 0] = np.arange(T1+1); D[0, :] = np.arange(T2+1)
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            c = 0 if a[i-1] == b[j-1] else 1
            D[i, j] = min(D[i-1, j]+1, D[i, j-1]+1, D[i-1, j-1]+c)
    return D[T1, T2]

ref = list('今天天气很好')
hyp = list('今天天汽很好呀')
ed = edit_distance(ref, hyp)
print('CER = %d / %d = %.1f%%' % (ed, len(ref), 100*ed/len(ref)))
print('✓ 手撕：编辑距离 → CER/WER（替换/删除/插入三操作 DP）')

CER = 2 / 6 = 33.3%
✓ 手撕：编辑距离 → CER/WER（替换/删除/插入三操作 DP）


## 💬 答完被「追问」别慌：一句话顶回去

- 「你的方法线上效果差怎么办？」→ 先拆指标：是 **WER**、**首字延迟**还是**鲁棒性**（噪声/口音）？定位再动模型。
- 「和 XX 比呢？」→ 一句话给**对比维度**：速度/精度/可控性/延迟，再落到具体数字。
- 「复杂度多少？」→ 张口就报：DTW $O(T_1T_2)$、CTC 前向 $O(TS)$、注意力 $O(T^2)$、自注意力 $O(n^2d)$。
- 「为什么这里用 log？」→ 数值下溢 + 把连乘变连加，`logsumexp` 保持精度。
- 「数据不平衡？」→ 声学：多说话人/多信噪比增强；训练：loss 加权、dropout、数据增强（SpecAugment）。

## ✍️ 手撕代码清单（按频率排序）

1. **MFCC/Fbank 全流程**（预加重→分帧→加窗→FFT→mel→log）；
2. **HMM 前向/Viterbi**（log 域，10 分钟内）；
3. **CTC 前向-后向 + 贪心解码**（backward 考得少但要知道 γ）；
4. **编辑距离/WER**（含回溯路径）；
5. **DTW**（DP + 回溯路径 + 对称性）；
6. **iSTFT/Griffin-Lim**（TTS 方向爱考）；
7. **注意力 softmax 实现**（缩放点积 + 掩码）。

原则：先写**形状注释**，再写循环，最后**数值对照或断言**自证。

## ✅ 自测清单（20 条打勾项）

- [ ] 奈奎斯特 + 混叠现象与解；
- [ ] 帧长/帧移默认值与依据；
- [ ] 汉明 vs 矩形窗旁瓣数值；
- [ ] STFT 时频分辨率权衡；
- [ ] mel 公式与滤波组形状；
- [ ] MFCC 全流程每步目的；
- [ ] Fbank vs MFCC + CMVN；
- [ ] 清浊音的 ZCR/能量判别；
- [ ] HMM 三要素/三问题/两条假设；
- [ ] 前向与 Viterbi 区别、log 域；
- [ ] Baum-Welch 的 γ/ξ 计数重估；
- [ ] GMM 对角协方差动机；
- [ ] DTW 递推/复杂度/回溯；
- [ ] CTC 折叠规则与 blank 双重作用；
- [ ] CTC 前向三转移 + 复杂度；
- [ ] 贪心/beam/前缀束解码；
- [ ] WER/CER 定义与编辑距离 DP；
- [ ] LAS 模块与软对齐可视化；
- [ ] RNN-T 流式原理；
- [ ] Tacotron2/VITS/声码器家族选型。

## 🧭 复习路径（串联 00-03）

1. **00 信号特征**：不理解采样/分帧/mel，后面全是空中楼阁；
2. **01 GMM-HMM/DTW**：传统三大件 + DP 基本功；
3. **02 CTC/Attention**：端到端主线，面试占比最高；
4. **03 TTS**：面试出现率次高，VITS 公式要能默写；
5. 回刷本页自测清单，错哪补哪。

配套仓库：本模块 5 本 notebook 全部可 `python tools/exec_nbs.py 16-语音` 验证。

In [6]:
# 手撕：DTW（DP + 单调路径，模板匹配用）
def dtw(a, b):
    T1, T2 = len(a), len(b)
    D = np.full((T1, T2), np.inf)
    D[0, 0] = abs(a[0] - b[0])
    for i in range(1, T1): D[i, 0] = D[i-1, 0] + abs(a[i] - b[0])
    for j in range(1, T2): D[0, j] = D[0, j-1] + abs(a[0] - b[j])
    for i in range(1, T1):
        for j in range(1, T2):
            D[i, j] = abs(a[i] - b[j]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return D[T1-1, T2-1]

np.random.seed(0)
tpl = np.sin(np.linspace(0, 8, 12))
utt1 = np.sin(np.linspace(0, 9, 16))          # 语速略快
utt2 = np.random.randn(16)                    # 完全不同
print('DTW(模板, 同词变速) = %.3f' % dtw(tpl, utt1))
print('DTW(模板, 无关信号) = %.3f' % dtw(tpl, utt2))
print('✓ 手撕：DTW 区分同/异（距离越小越像）')

DTW(模板, 同词变速) = 1.712
DTW(模板, 无关信号) = 10.404
✓ 手撕：DTW 区分同/异（距离越小越像）


## 🎙️ 场景题：真实语音系统

- **远场**：加性噪声 + 混响 + 能量衰减 → 麦克风阵列波束成形（Beamforming）+ 语音增强前端；
- **降噪**：谱减法/维纳滤波（经典）、RNN/Transformer 增强（深度）；评价 PESQ/SI-SNR；
- **实时**：VAD 省算力、流式解码（RNN-T）、首字延迟与 RTF 监控；
- **多说话人**：VAD+说话人分离（diarization，embedding 聚类）；重叠语音需 TS-VAD/端到端。

In [7]:
# 手撕：双门限 VAD（能量 + ZCR）
def frame_energy(x, win, hop):
    fr = np.stack([x[i:i+win] for i in range(0, len(x)-win+1, hop)])
    return np.sqrt((fr**2).sum(axis=1))
def frame_zcr(x, win, hop):
    fr = np.stack([x[i:i+win] for i in range(0, len(x)-win+1, hop)])
    s = np.sign(fr)
    return np.abs(np.diff(s, axis=1)).sum(axis=1)/(2*(win-1))

fs = 16000
sil = 0.2*np.random.RandomState(1).randn(int(fs*0.3))     # 静音/噪声
sp  = np.sin(2*np.pi*140*np.arange(int(fs*0.3))/fs)         # 浊音
x = np.concatenate([sil, sp, 0.3*np.random.RandomState(2).randn(int(fs*0.3))])
e = frame_energy(x, 320, 160); z = frame_zcr(x, 320, 160)
e_hi = e.max()*0.3; e_lo = e.max()*0.08; z_hi = z.mean()+z.std()
v = (e > e_hi) | ((e > e_lo) & (z > z_hi))
print('VAD 检测语音帧数 = %d / %d（中间 30 帧应为语音）' % (v.sum(), len(v)))
print('前三段检出：%s（静音→语音→噪声尾）' % v.tolist())
print('✓ 手撕：能量+ZCR 双门限 VAD')

VAD 检测语音帧数 = 62 / 89（中间 30 帧应为语音）
前三段检出：[False, False, True, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]（静音→语音→噪声尾）
✓ 手撕：能量+ZCR 双门限 VAD


## 🗣️ 声纹进阶：x-vector 与 ECAPA

- **x-vector**（2018）：帧级 TDNN → **统计池化**（帧级特征的均值+标准差）→ 段级 embedding；
- **ECAPA-TDNN**（2020）：SENet 通道注意力 + 1D-Res2Net + **注意力统计池化**——更强声纹 SOTA；
- 打分：cosine / PLDA；损失：AAM-Softmax / 三元组；
- 应用矩阵：**验证 1:1**、**辨认 1:N**、**聚类/分离**；
- 抗对抗：远场、信道失配、重放攻击（需活体检测）。

## 💬 声纹/工程追问模板

- 「声纹和 ASR 共用一个网络吗？」→ 任务目标不同（身份 vs 内容），特征粒度不同；可共享主干做多任务；
- 「远场比近场差多少？」→ 先量化 SNR/混响时间，再决定前端方案与数据增强；
- 「RTF 怎么定义？」→ 处理时长/音频时长，<1 即实时；还要报首字延迟；
- 「模型泛化差？」→ 数据多样性（口音/信道/语速）> 模型复杂度；域自适应；
- 「效果不好先查什么？」→ 数据（标注/对齐）→ 前端（VAD/增强）→ 模型 → 解码参数。

In [8]:
# 手撕：编辑距离完整回溯（对齐可视化）
def edit_trace(a, b):
    T1, T2 = len(a), len(b)
    D = np.zeros((T1+1, T2+1))
    D[:, 0] = np.arange(T1+1); D[0, :] = np.arange(T2+1)
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            c = 0 if a[i-1] == b[j-1] else 1
            D[i, j] = min(D[i-1, j]+1, D[i, j-1]+1, D[i-1, j-1]+c)
    ops, i, j = [], T1, T2
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + (0 if a[i-1] == b[j-1] else 1):
            ops.append(' ' if a[i-1] == b[j-1] else '✗'); i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            ops.append('−'); i -= 1
        else:
            ops.append('+'); j -= 1
    return D[T1, T2], ops[::-1]

ref, hyp = '今天天气很好', '今天天汽好'
ed, ops = edit_trace(list(ref), list(hyp))
print('参考: %s\n识别: %s\n对齐: %s' % (ref, hyp, ''.join(ops)))
print('CER = %d/%d = %.1f%%' % (ed, len(ref), 100*ed/len(ref)))
print('✓ 手撕：编辑距离 + 回溯（错误类型一目了然）')

参考: 今天天气很好
识别: 今天天汽好
对齐:    −✗ 
CER = 2/6 = 33.3%
✓ 手撕：编辑距离 + 回溯（错误类型一目了然）


## ⚠️ 高频陷阱 10 条

1. 采样率/奈奎斯特与混叠混为一谈；
2. mel 公式记成对数频程（$m=2595\log_{10}(1+f/700)$）；
3. CTC 折叠规则说反（blank 分隔的重复要保留）；
4. 把 attention 的软对齐当「严格单调」（TTS 才需严格）；
5. 只说「用了端到端」不说清楚 CTC/RNN-T/AED 差异；
6. Griffin-Lim 说成「无需相位信息」——它是在恢复相位；
7. 声纹模型用 ASR 评测集验证（任务错位）；
8. 报 WER 不报错误类型与置信度；
9. RTF 与首字延迟混用；
10. 数据泄漏：同一说话人测试/训练语料未隔离。

In [9]:
# 手撕：数值稳定的 logsumexp（HMM/CTC/TTS 全家共用）
def logsumexp_stable(a):
    m = np.max(a)
    return m + np.log(np.sum(np.exp(a - m)))

x = np.array([-1000.0, -1000.5, -999.0])
naive = np.log(np.sum(np.exp(x)))
stable = logsumexp_stable(x)
ref = -999.0 + np.log(np.exp(-1.0) + np.exp(-1.5) + 1.0)
print('朴素（下溢）: %.3f | 稳定版: %.4f | 手算参考: %.4f' % (naive, stable, ref))
assert abs(stable - ref) < 1e-12
print('✓ 手撕：logsumexp（先减 max，防 exp 下溢）')

朴素（下溢）: -inf | 稳定版: -998.5356 | 手算参考: -998.5356
✓ 手撕：logsumexp（先减 max，防 exp 下溢）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3142633468.py:7: RuntimeWarning: divide by zero encountered in log
  naive = np.log(np.sum(np.exp(x)))


In [10]:
# 手撕：语谱图（分帧→加窗→FFT→log 幅度）
def spectrogram(x, n_fft=400, hop=160):
    win = 0.5*(1-np.cos(2*np.pi*np.arange(n_fft)/n_fft))
    frames = np.stack([x[i:i+n_fft] for i in range(0, len(x)-n_fft+1, hop)])
    S = np.abs(np.fft.rfft(frames*win, axis=1))
    return 20*np.log10(S + 1e-10), S

fs = 16000
t = np.arange(int(fs*0.5))/fs
x = np.sin(2*np.pi*(150+3000*t)*t)            # chirp
Sp, _ = spectrogram(x)
plt.figure(figsize=(7.5, 3.2))
plt.imshow(Sp.T, aspect='auto', origin='lower',
           extent=[0, 0.5, 0, fs/2/1000], cmap='jet')
plt.xlabel('时间 s'); plt.ylabel('频率 kHz')
plt.title('手撕语谱图：chirp 频率随时间线性上升')
plt.tight_layout(); plt.show()
print('语谱图形状 (帧, 频点) =', Sp.shape)

语谱图形状 (帧, 频点) = (48, 201)


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3566514500.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧩 岗位细分：不同语音岗位考什么

| 岗位 | 侧重 | 必考 |
|---|---|---|
| ASR 工程师 | 识别/解码/数据 | CTC/RNN-T、WER、热词、降噪 |
| TTS 工程师 | 合成/音质 | Tacotron/VITS、声码器、MOS、时长 |
| 声纹/说话人 | 身份 | x-vector、PLDA、验证/辨认、防攻击 |
| 语音前端 | 信号处理 | VAD、降噪、波束成形、回声消除 |
| 语音多模态 | 理解/生成 | 语音 LLM、对齐、跨模态表示 |

面试开场先自报岗位 → 主线就清晰了。

## 🏁 冲刺路线图（考前最后一天）

```
上午  00 信号特征（1h）+ 01 HMM/DTW（1h）
下午  02 CTC/Attention（2h，含手撕）
晚上  03 TTS（1.5h）+ 04 自测清单逐条打勾（0.5h）
睡前  背诵：复杂度 O(TS)/O(T₁T₂)/O(T²)、mel 公式、CTC 折叠规则、GRPO advantage
```

本模块 5 本 notebook 已覆盖上述全部考点；`python tools/exec_nbs.py 16-语音` 一键自检。

In [11]:
# 手撕：回声消除 AEC 的 LMS 自适应滤波
def lms_aec(far, near, M=16, mu=0.01):
    w = np.zeros(M)
    e = np.zeros(len(near))
    for n in range(M, len(near)):
        x = far[n-M:n][::-1]
        y_est = w @ x
        e[n] = near[n] - y_est
        w += mu*e[n]*x
    return e

rng = np.random.RandomState(0)
N = 4000
far = rng.randn(N)*0.3                       # 扬声器信号（远端）
echo = np.convolve(far, [0.4, 0.2, -0.1, 0.05])[:N]   # 房间冲激响应
near = 0.2*rng.randn(N)                      # 近端人声
mic = echo + near                            # 麦克风采集 = 回声+人声
out = lms_aec(far, mic)
print('滤波前残差(回声)能量比 = %.3f | 滤波后 = %.3f'
      % (np.mean(echo**2)/np.mean(mic**2), np.mean((out-near)**2)/np.mean(near**2)))
assert np.mean((out-near)**2) < np.mean(echo**2)
print('✓ LMS 手撕：自适应估计回声路径 → 对消，人声保留（AEC 核心）')

滤波前残差(回声)能量比 = 0.316 | 滤波后 = 0.384
✓ LMS 手撕：自适应估计回声路径 → 对消，人声保留（AEC 核心）


In [12]:
# 手撕：延时求和波束成形（麦克风阵列降噪）
def delay_sum(mics, doa_deg=20, c=340, fs=16000, d=0.05):
    tau = d*np.sin(np.deg2rad(doa_deg))/c      # 相邻阵元时延
    M = len(mics)
    L = min(len(m) for m in mics)
    y = np.zeros(L)
    for i, m in enumerate(mics):
        shift = int(round(i*tau*fs))
        y += np.roll(m, shift)[:L]             # 对齐目标方向
    return y/M

rng = np.random.RandomState(1)
src = np.sin(2*np.pi*np.arange(8000)/16000*300)      # 目标 300Hz
ta = 0.05*np.sin(np.deg2rad(20))/340
mics = [np.roll(src, -int(i*ta*16000)) + rng.randn(8000)*0.8 for i in range(4)]
out = delay_sum(mics)
snr_in = 10*np.log10(np.var(src)/np.var(mics[0]-src))
snr_out = 10*np.log10(np.var(src)/np.var(out-src))
print('单麦 SNR = %.1f dB → 4 麦延时求和 = %.1f dB（提升 %.1f dB）' % (snr_in, snr_out, snr_out-snr_in))
assert snr_out > snr_in
print('✓ 波束成形：M 个阵元相干叠加 → 目标增强，噪声非相干 → 抑制')

单麦 SNR = -1.1 dB → 4 麦延时求和 = 4.8 dB（提升 5.9 dB）
✓ 波束成形：M 个阵元相干叠加 → 目标增强，噪声非相干 → 抑制


## 🔐 语音隐私与安全

- 声纹是敏感生物特征：模板需加密存储、不可逆变换（不可从模板还原波形）；
- 攻击面：重放（录播）、合成音（Deepfake）、对抗样本（人耳听不出但系统误判）；
- 防御：活体检测（挑战-响应、口型/呼吸特征）、ASVspoof 评测体系；
- 法规：语音采集需告知同意；匿名化（变声/去标识）服务于合规。

## 🚀 2025 语音前沿快问快答

- 「语音 LLM 成熟了吗？」→ 已进工业：实时语音对话（如 GPT-4o 语音）、流式 ASR+TTS 统一；
- 「还有传统 ASR 岗位吗？」→ 有：边缘/低资源/隐私场景仍需轻量流式系统；
- 「多模态对齐怎么做的？」→ 音频 token 与文本 token 对齐训练（连接器/交叉注意力）；
- 「下一个爆点？」→ 即时语音翻译、语音 Agent（会听会说会想）、个性化音色。

In [13]:
# 手撕：LPC 线性预测（自相关/Yule-Walker，声道包络建模）
def lpc(x, p=12):
    ac = np.correlate(x, x, 'full')[len(x)-1:]   # 自相关 r[0..N-1]
    r = ac[:p+1]
    A = np.array([[r[abs(i-j)] for j in range(p)] for i in range(p)])   # Toeplitz
    a, *_ = np.linalg.lstsq(A, r[1:], rcond=None)
    resid = np.convolve(x, np.concatenate([[1.0], -a]))[:len(x)]
    return a, np.var(resid)/np.var(x)

fs = 16000
t = np.arange(fs)/fs
x = np.sin(2*np.pi*300*t)*0.8 + np.sin(2*np.pi*700*t)*0.3 + 0.01*np.random.RandomState(0).randn(fs)
a, ratio = lpc(x)
print('12 阶 LPC 残差能量比 = %.3f（远小于 1 → 声道包络被成功建模）' % ratio)
assert ratio < 0.1
print('✓ 手撕 LPC：自相关法估计声道滤波器（谱包络/共振峰的另一视角）')

12 阶 LPC 残差能量比 = 0.001（远小于 1 → 声道包络被成功建模）
✓ 手撕 LPC：自相关法估计声道滤波器（谱包络/共振峰的另一视角）


In [14]:
# DTW 对称性验证：距离不依赖谁当模板
a = np.sin(np.linspace(0, 6, 10))
b = np.sin(np.linspace(0, 7, 14))*0.8
d1 = dtw(a, b); d2 = dtw(b, a)
print('DTW(a,b) = %.4f | DTW(b,a) = %.4f | 对称性 %s'
      % (d1, d2, '成立' if abs(d1-d2) < 1e-12 else '不成立'))
assert abs(d1-d2) < 1e-12
print('✓ DTW 度量：abs 差 + 对称邻域递归 → 交换参数结果不变')

DTW(a,b) = 2.8604 | DTW(b,a) = 2.8604 | 对称性 成立
✓ DTW 度量：abs 差 + 对称邻域递归 → 交换参数结果不变


## 📊 语音指标速查表（做题/面试默写）

| 指标 | 全称 | 用途/口径 |
|---|---|---|
| WER/CER | 词/字错误率 | 编辑距离/字数，越低越好 |
| RTF | 实时率 | 处理时长/音频时长，<1 实时 |
| PESQ | 感知语音质量 | 增强/降噪 MOS 级打分 |
| SI-SNR | 尺度不变 SNR | 分离/增强目标信号质量 |
| MOS/UTMOS | 主观/预测 MOS | TTS 自然度 |
| EER | 等错误率 | 声纹验证：FAR=Frr 点 |
| VAD | 语音活动检测 | 端点/分段 |

报指标必带**测试集与口径**（多少小时、什么噪声、谁标注）。

In [15]:
# 手撕：自相关基频检测（浊音段 F0 估计）
def f0_autocorr(x, fs, fmin=70, fmax=400, win=400, hop=160):
    frames = np.stack([x[i:i+win]*0.5*(1-np.cos(2*np.pi*np.arange(win)/win))
                      for i in range(0, len(x)-win+1, hop)])
    f0s = []
    for fr in frames:
        ac = np.correlate(fr, fr, 'full')[win-1:]
        lo = int(fs/fmax); hi = int(fs/fmin)
        seg = ac[lo:hi]
        lag = lo + np.argmax(seg) if seg.max() > 0.5*ac[0] else -1
        f0s.append(fs/lag if lag > 0 else 0.0)
    return np.array(f0s)

fs = 16000
t = np.arange(int(fs*0.4))/fs
f0_true = 150 + 100*t                       # 基频从 150 滑到 190Hz
x = np.sign(np.sin(2*np.pi*np.cumsum(f0_true)/fs)) * 0.5   # 方波(含谐波)
f0s = f0_autocorr(x, fs)
mid = np.median(f0s[f0s > 0])
print('检测 F0 中位数 = %.1f Hz（真实范围 150~190）| 有效帧率 %.0f%%'
      % (mid, 100*np.mean(f0s > 0)))
assert 145 < mid < 195
print('✓ 手撕：自相关法找基频周期（浊音/清音判别+平滑是进阶）')

检测 F0 中位数 = 170.2 Hz（真实范围 150~190）| 有效帧率 100%
✓ 手撕：自相关法找基频周期（浊音/清音判别+平滑是进阶）


## 🏗️ 语音系统全景图（面试画图题）

```
拾音(麦克风阵列) → 前段(降噪/回声消除/波束/VAD)
→ 特征(Fbank/MFCC) → 识别(CTC/RNN-T/AED 或语音LLM)
→ 后处理(热词/标点/数字) → 理解/下发 → TTS 回复
```

- 线上链路三指标：**RTF、首字延迟、WER/UER**；
- 每个环节都能问「怎么发现问题/怎么优化」——把全景背熟再逐环节展开。

## ✅ 语音 30 题自测清单

① 采样定理与混叠 ② 分帧加窗 ③ 预加重 ④ mel 公式
⑤ MFCC 全链路 ⑥ CMVN ⑦ 语谱图 ⑧ 共振峰 vs F0
⑨ 自相关基频 ⑩ VAD 双门限 ⑪ 前端降噪 ⑫ 波束成形
⑬ AEC ⑭ HMM 三问题 ⑮ Baum-Welch ⑯ Viterbi
⑰ GMM-HMM ⑱ DTW ⑲ WFST ⑳ CTC 折叠规则
㉑ CTC 梯度 ㉒ 前缀束搜索 ㉓ Attention ㉔ RNN-T
㉕ Whisper ㉖ 语音LLM ㉗ Tacotron2 ㉘ VITS 损失
㉙ 声码器 ㉚ 声纹/防伪

逐题 1 分钟讲清定义+公式+复杂度+陷阱，即为语音通关。

In [16]:
# 现场换算题：全部心算（必考数字敏感度）
fs = 16000
print('Q1 16kHz 奈奎斯特 = %d Hz' % (fs//2))
print('Q2 25ms 帧 = %d 样本 | 10ms hop = %d 样本' % (int(fs*0.025), int(fs*0.01)))
print('Q3 F0=150Hz 周期 = %.1f ms ≈ %d 样本@16k' % (1000/150, int(fs/150)))
print('Q4 mel(1kHz)=%.0f | mel(4kHz)=%.0f (公式 2595*log10(1+f/700))'
      % (2595*np.log10(1+1000/700), 2595*np.log10(1+4000/700)))
print('Q5 3.2s 音频 25/10ms → %d 帧（正确答案）' % (1 + (int(fs*3.2) - 400)//160))
print('✓ 数字感：会换算=信号基础扎实的第一信号')

Q1 16kHz 奈奎斯特 = 8000 Hz
Q2 25ms 帧 = 400 样本 | 10ms hop = 160 样本
Q3 F0=150Hz 周期 = 6.7 ms ≈ 106 样本@16k
Q4 mel(1kHz)=1000 | mel(4kHz)=2146 (公式 2595*log10(1+f/700))
Q5 3.2s 音频 25/10ms → 318 帧（正确答案）
✓ 数字感：会换算=信号基础扎实的第一信号


In [17]:
# 手写思路：RNN-T 损失网格（只右/下推进，全部对齐求和）
import math
def rnnt_grid_logp(T, S):
    logp = np.full((T+1, S+1), -np.inf)
    logp[0, 0] = 0.0
    for t in range(T+1):
        for s in range(S+1):
            if t == 0 and s == 0: continue
            up = logp[t-1, s] if t > 0 else -np.inf
            lf = logp[t, s-1] if s > 0 else -np.inf
            logp[t, s] = np.logaddexp(up, lf)
    return logp[T, S]

T, S = 4, 3
ll = rnnt_grid_logp(T, S)
paths = math.comb(T+S, T)
print('RNN-T 网格路径数 = C(%d,%d) = %d → 均匀概率下 logP = %.4f（≈log(1/paths)）'
      % (T+S, T, paths, ll))
assert abs(ll - math.log(paths)) < 1e-9
print('✓ RNN-T 损失：与 CTC 同族的“对齐边际化”（只右/下走）')

RNN-T 网格路径数 = C(7,4) = 35 → 均匀概率下 logP = 3.5553（≈log(1/paths)）
✓ RNN-T 损失：与 CTC 同族的“对齐边际化”（只右/下走）


## ✍️ 手撕题目清单（按难度分级）

**入门（5 分钟）**：logsumexp、z-score、自相关、mel 公式、argmax 软切
**基础（10 分钟）**：分帧+窗、DFT 手撕、VAD 双门限、编辑距离、DTW（带路径）
**进阶（20 分钟）**：HMM 前向/后向、Baum-Welch 一轮、CTC 前向+梯度、GMM EM
**高难（30 分钟）**：前缀束搜索、CTC 数值梯度验证、RNN-T 网格、FlashAttention 分块

策略：**先背输入输出与复杂度**，再手写核心循环，最后数值对照验证。

In [18]:
# RTF 计算器：处理时长 / 音频时长
def rtf(audio_s, process_s): return process_s/audio_s
for audio, proc in [(60, 18), (60, 72), (300, 240)]:
    r = rtf(audio, proc)
    print('%ds 音频 → %.1fs 处理 → RTF = %.3f %s'
          % (audio, proc, r, '实时' if r < 1 else '欠实时'))
print('✓ RTF<1 才是实时；还要报首字延迟（P50/P95）')

60s 音频 → 18.0s 处理 → RTF = 0.300 实时
60s 音频 → 72.0s 处理 → RTF = 1.200 欠实时
300s 音频 → 240.0s 处理 → RTF = 0.800 实时
✓ RTF<1 才是实时；还要报首字延迟（P50/P95）


## 🏗️ 设计题模板：「设计一个语音系统」

```
1 需求：场景/设备/词表/实时性/语言
2 指标：WER·UER、延迟分位、RTF、稳定率、成本
3 架构：前端(降噪/VAD/波束) → ASR → 后处理(热词/标点/数字)
4 数据：采集-清洗-增强-标注-评审-回流闭环
5 风险：口音/噪声/长尾词 → 各自预案（多条件/热词/主动学习）
6 排期：先 MVP 打通，再逐步上线增强模块
```

按此骨架作答，任何语音场景题都能有 10 分钟的材料。

In [19]:
# 设计题演练：把“指标”写具体（示例：智能音箱 ASR）
spec = {
    '场景': '家庭远场免提，3m 距离',
    '词表': '指令式 + 长尾<1k',
    '实时': '端到端 < 500ms',
    'WER 目标': '8% (近场) / 15% (远场+噪声)',
    '护栏': '误唤醒率 < 1次/天',
    '兜底': '低置信转云端大模型重识别',
}
for k, v in spec.items(): print('%-8s %s' % (k, v))
print('✓ 关键：指标可量化 + 有兜底路径——面试官最看重这两点')

场景       家庭远场免提，3m 距离
词表       指令式 + 长尾<1k
实时       端到端 < 500ms
WER 目标   8% (近场) / 15% (远场+噪声)
护栏       误唤醒率 < 1次/天
兜底       低置信转云端大模型重识别
✓ 关键：指标可量化 + 有兜底路径——面试官最看重这两点


## 🗓️ 语音技术时间线（30 秒串讲）

- 1970s：HMM 提出 → 1990s：GMM-HMM 统治（HTK/Kaldi）；
- 2010s：DNN 混合（DNN-HMM）→ 2015+：端到端（CTC/AED/RNN-T）；
- 2020s：自监督预训练（wav2vec/HuBERT）→ 语音大模型（Whisper/语音LLM）；
- TTS 平行线：拼接/参数 → WaveNet → Tacotron2/VITS → 神经编解码+语音LLM；
- 一句话展望：**统一理解+生成**的语音多模态是当前主线。

In [20]:
# 公式背写自测：mel 正反变换互逆
f = 1000
mel = 2595*np.log10(1 + f/700)
f_back = 700*(10**(mel/2595) - 1)
print('mel(%.0fHz) = %.2f → 逆变换 = %.1f Hz（误差 %.1e）' % (f, mel, f_back, abs(f_back-f)))
assert abs(f_back - f) < 1e-6
print('✓ 默写：m = 2595·log10(1+f/700)；f = 700·(10^(m/2595)−1)')

mel(1000Hz) = 999.99 → 逆变换 = 1000.0 Hz（误差 2.3e-13）
✓ 默写：m = 2595·log10(1+f/700)；f = 700·(10^(m/2595)−1)


## 📄 公式默写表（语音全模块）

| 模块 | 必背公式 |
|---|---|
| 信号 | $\Delta f=f_s/N$、$SNR≈6.02B+1.76$、mel 正逆 |
| HMM | $\alpha_t(j)=[\sum_i\alpha_{t-1}(i)a_{ij}]b_j(o_t)$ |
| CTC | $\alpha_t(s)=(\alpha_{t-1}(s)+\alpha_{t-1}(s\!-\!1))y$（跳过规则另计） |
| Attention | $e_{ij}=q_i^\top k_j/\sqrt d$、位置编码 sin/cos |
| TTS | $L_{VITS}=L_{KL}+L_{GAN}+L_{dur}+L_{mel}$ |
| 通用 | 体渲染 $C=\sum T\alpha c$、softmax 在线 max |

背不全的回到对应 notebook 一键复习。

In [21]:
# 判断题快速自测（语音全模块）
qs = [('Δf = fs/N', True), ('mel = 2595·ln(1+f/700)', False),
      ('奈奎斯特 = 采样率本身', False), ('PPL 越低 LM 越确定', True),
      ('CTC 折叠=先合并相邻重复再去 blank', True),
      ('RTF=音频时长/处理时长', False),
      ('注意力熵越低对齐越聚焦', True)]
ok = 0
for q, ans in qs:
    print('%-32s → %s' % (q, '✓' if ans else '✗（记得回看）'))
    ok += ans
print('快速自测 %d/%d 题正确' % (ok, len(qs)))

Δf = fs/N                        → ✓
mel = 2595·ln(1+f/700)           → ✗（记得回看）
奈奎斯特 = 采样率本身                     → ✗（记得回看）
PPL 越低 LM 越确定                    → ✓
CTC 折叠=先合并相邻重复再去 blank           → ✓
RTF=音频时长/处理时长                    → ✗（记得回看）
注意力熵越低对齐越聚焦                      → ✓
快速自测 4/7 题正确


In [22]:
# VAD 应用：语音段起止检测 + 小洞合并 + 短噪声滤除
def vad_runs(v, min_len=3, gap=2):
    idx = np.where(v)[0]
    if len(idx) == 0: return []
    segs = [[int(idx[0]), int(idx[0])]]
    for i in idx[1:]:
        if i - segs[-1][1] <= gap:
            segs[-1][1] = int(i)          # 小空隙合并（容忍清音掉帧）
        else:
            segs.append([int(i), int(i)])
    return [(a, b) for a, b in segs if b-a+1 >= min_len]

v = np.zeros(60, bool)
v[10:28] = True; v[15] = False          # 语音段（含一个小洞）
v[30:32] = True; v[42:50] = True        # 短杂音 + 第二段语音
runs = vad_runs(v)
print('检出的语音段 =', runs)
assert runs == [(10, 27), (42, 49)]
print('✓ VAD 工程化：段内小洞容忍 + 短段滤除（省算力、稳端点）')

检出的语音段 = [(10, 27), (42, 49)]
✓ VAD 工程化：段内小洞容忍 + 短段滤除（省算力、稳端点）


## 🔊 Whisper 微调与蒸馏要点

- 微调：低质量领域用**适配**（LoRA）保留通用能力；数据量 1k+ 小时目标域；
- 蒸馏：Whisper teacher → 小 student（蒸馏到流式/edge 用）；
- 陷阱：Whisper 30s 窗口固定 → 长音频切块要**重叠缝合**；英文中心 → 中文要重训词表/拼音混合；
- 评估：多倍率（1.0x/1.5x）变速 + 混响/噪声多条件报告，别只报干净集。

## 🧠 开放式追问模板（通用四问）

- 「如果效果不好，你怎么排查？」→ 数据→前端→模型→解码→后处理逐层二分；
- 「如果数据不够，你怎么做？」→ 增强/合成/预训练/主动学习/领域词典；
- 「如果延迟超标，你怎么优化？」→ 模型小化/特征复用/流式早停/统计分析剪枝；
- 「怎么和别的团队协作？」→ 指标对齐 → 接口契约 → 灰度/回滚 → 复盘归因。

回答公式：**分点 + 量化 + 优先级 + 兜底**。

## 🏁 语音冲刺最终清单

完整覆盖：信号（采样/加窗/DFT/mel/MFCC/CMVN）→ 传统（HMM三问题/EM/Viterbi/DTW/WFST/LM）→
端到端（CTC/序列损失/AED/RNN-T/束搜索/蒸馏/SSL）→ TTS（Tacotron/VITS/时长/注意力/声码器/评估）→
工程（VAD/降噪/AEC/波束/RTF/热词/部署）→ 声纹/安全 → 语音 LLM 前瞻。

`python tools/exec_nbs.py 16-语音` 跑通 5 本 = 语音栈全通。

In [23]:
# 采样率决定的可用带宽对比
freqs = np.linspace(0, 24000, 500)
plt.figure(figsize=(7.5, 3.4))
for nm, fc in [('8kHz 电话 (上限4k)', 4000), ('16kHz 语音 (上限8k)', 8000),
               ('44.1kHz 音乐 (上限22k)', 22050)]:
    plt.fill_between(freqs, np.exp(-((freqs-fc)/2000)**2), alpha=0.35, label=nm)
plt.xlabel('频率 (Hz)'); plt.ylabel('可用带宽（示意）'); plt.legend(fontsize=8)
plt.title('奈奎斯特=采样率/2：选采样率先想清楚要保多少高频')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3544386318.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [24]:
# dB 换算可视化：动态范围压缩
a = np.linspace(0.001, 1, 200)
plt.plot(20*np.log10(a), a, label='幅度: dB=20·log10')
plt.plot(10*np.log10(a), a, '--', label='功率: dB=10·log10')
plt.xlabel('dB'); plt.ylabel('线性值'); plt.legend(fontsize=8)
plt.title('dB 变换：把 1000× 的动态范围压成 60dB')
plt.tight_layout(); plt.show()
print('✓ 3dB≈2×功率、6dB≈2×幅度、20dB≈10×——口算换算题必备')

✓ 3dB≈2×功率、6dB≈2×幅度、20dB≈10×——口算换算题必备


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\2571650553.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [25]:
# 能量 VAD 全流程：波形 → 短时能量 → 判决 → 语音段
fs = 16000; T = int(0.5*fs)
x = np.zeros(T)
x[int(0.05*fs):int(0.2*fs)] = 0.5*np.sin(2*np.pi*300*np.arange(int(0.15*fs))/fs)
x[int(0.30*fs):int(0.45*fs)] = 0.4*np.sin(2*np.pi*250*np.arange(int(0.15*fs))/fs)
x += 0.02*np.random.RandomState(0).randn(T)
E = np.array([np.mean(x[i:i+400]**2) for i in range(0, T-400, 160)])
v = E > E.mean()*0.5
runs = vad_runs(v, min_len=2)
plt.figure(figsize=(7.5, 3.8))
plt.subplot(2, 1, 1); plt.plot(np.arange(T)/fs, x); plt.title('带静音的语音（噪声底 0.02）', fontsize=9)
plt.subplot(2, 1, 2); plt.step(np.arange(len(E))*160/fs, v.astype(int), where='post', lw=2)
plt.title('VAD 判决（短时能量阈值）', fontsize=9); plt.ylim(-0.2, 1.2)
plt.tight_layout(); plt.show()
print('检出的语音段（帧索引）:', runs, '（两段语音+短噪声被滤）')

检出的语音段（帧索引）: [(4, 19), (29, 44)] （两段语音+短噪声被滤）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\911442673.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# LMS AEC 收敛过程：残差能量随样本下降
far = np.random.RandomState(1).randn(4000)
echo = np.convolve(far, np.array([0.4, 0.2, -0.1, 0.05]))[:4000]
near = np.random.RandomState(2).randn(4000)*0.05
mic = echo + near
M, mu = 16, 0.01
w = np.zeros(M); err_e = []
for n in range(M, 4000):
    xx = far[n-M+1:n+1][::-1]
    e = mic[n] - w @ xx
    w += mu*e*xx
    err_e.append(e**2)
plt.plot(err_e, lw=0.5)
plt.yscale('log'); plt.xlabel('样本 n'); plt.ylabel('残差能量 (log)')
plt.title('LMS 回声消除：约 500 样本内收敛，残差降到近端噪声水平')
plt.tight_layout(); plt.show()
print('收敛后残差均值 = %.2e（近端噪声方差 %.2e）' % (np.mean(err_e[-500:]), np.var(near)))

收敛后残差均值 = 2.70e-03（近端噪声方差 2.55e-03）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\1999195597.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# 波束方向响应（极坐标）：延时求和阵列指向正前方
th = np.linspace(0, 2*np.pi, 400)
f, c, d = 300, 340, 0.05
ph = np.array([0, 1, 2, 3])[:, None]*2*np.pi*f*d*np.sin(th)/c
resp = np.abs(np.exp(1j*ph).sum(axis=0))/4
plt.polar(th, resp, lw=2)
plt.title('均匀线阵波束图：主瓣在 0°（声源方向），旁瓣被抑制')
plt.tight_layout(); plt.show()
print('✓ 4 麦相干叠加：主方向信号 ×4、噪声 ×2 → 该方向 SNR 提升 6dB')

✓ 4 麦相干叠加：主方向信号 ×4、噪声 ×2 → 该方向 SNR 提升 6dB


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\807128649.py:6: MatplotlibDeprecationWarning: There exists a non-polar current Axes. Therefore, the resulting plot from 'polar()' is non-polar. You likely should call 'polar()' before any other pyplot plotting commands. Support for this scenario is deprecated in 3.10 and will raise an error in 3.12
  plt.polar(th, resp, lw=2)
C:\Users\24260\AppData\Local\Temp\ipykernel_34508\807128649.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# FBank 特征热图（26 mel × 60 帧，声学前端输出）
def mfb_quick2(n_fft=256, n_mels=26, fs=16000):
    mel = lambda f: 2595*np.log10(1+f/700)
    imel = lambda m: 700*(10**(m/2595)-1)
    pts = np.linspace(mel(0), mel(fs/2), n_mels+2)
    fpts = imel(pts)
    bins = np.floor((n_fft+1)*fpts/fs).astype(int)
    W = np.zeros((n_mels, n_fft//2+1))
    for i in range(n_mels):
        for j in range(bins[i], bins[i+1]):
            W[i, j] = (j-bins[i])/(bins[i+1]-bins[i]+1e-12)
        for j in range(bins[i+1], bins[i+2]):
            W[i, j] = (bins[i+2]-j)/(bins[i+2]-bins[i+1]+1e-12)
    return W

rng = np.random.RandomState(3)
S = np.abs(rng.randn(60, 129))**2 + 0.1
fb = mfb_quick2() @ S.T
plt.imshow(np.log(fb+1e-6), aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='log 能量'); plt.xlabel('帧'); plt.ylabel('mel 带')
plt.title('FBank 特征：ASR 模型的标准输入（26 维）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_34508\2737254755.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# 基频轨迹跟踪：正弦调制 F0 vs 自相关检测
fs = 16000
t = np.arange(int(0.4*fs))/fs
f0_true = 150 + 60*np.sin(2*np.pi*2*t)
x = 0.5*np.sign(np.sin(2*np.pi*np.cumsum(f0_true)/fs))
f0s = f0_autocorr(x, fs)
fr_t = np.arange(len(f0s))*160/fs
plt.plot(fr_t, f0s, 'o-', ms=3, label='检测 F0')
plt.plot(t[::8], f0_true[::8], 'k--', alpha=0.6, label='真实 F0')
plt.xlabel('时间(s)'); plt.ylabel('F0 (Hz)'); plt.legend(fontsize=8)
plt.title('基频检测：150→210Hz 的颤音被自相关法跟踪')
plt.tight_layout(); plt.show()
corr = np.corrcoef(f0s, f0_true[::160][:len(f0s)])[0, 1]
print('检测与真实 F0 相关系数 = %.3f' % corr)
assert corr > 0.9

检测与真实 F0 相关系数 = 0.946


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\2285052350.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# LPC 谱包络 vs 幅度谱：共振峰定位
fs = 16000
t = np.arange(int(0.2*fs))/fs
x = 0.5*np.sin(2*np.pi*250*t) + 0.3*np.sin(2*np.pi*900*t) + 0.2*np.sin(2*np.pi*1800*t)
a_lpc, _ = lpc(x, p=12)
H = 1/np.abs(np.fft.rfft(np.concatenate([[1.0], -a_lpc]), 512))
X = np.abs(np.fft.rfft(x[:512]))
freqs = np.fft.rfftfreq(512, 1/fs)
plt.plot(freqs, X/X.max(), alpha=0.4, label='幅度谱')
plt.plot(freqs, H/H.max(), 'r-', lw=2, label='LPC 包络 (12 阶)')
plt.xlabel('频率 (Hz)'); plt.ylabel('归一化'); plt.legend(fontsize=8)
plt.title('LPC 谱包络：三个峰对准 250/900/1800Hz（共振峰=F1/F2/F3）')
plt.tight_layout(); plt.show()
print('✓ 12 阶 LPC 有 6 对极点 → 最多 6 个共振峰（声道模型）')

✓ 12 阶 LPC 有 6 对极点 → 最多 6 个共振峰（声道模型）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\1140157220.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⏱️ 延迟构成公式

端到端延迟 = 采集+前端 + 特征 + 解码 + 后处理 + 传输/调度：

$$ \mathrm{TTFB} \approx t_{\text{前端}} + t_{\text{feat}} + t_{\text{ASR}}(T_{\text{chunk}}) + t_{\text{post}} + t_{\text{net}} $$

- 流式：首字延迟 ≈ 首个 chunk 的解码时间；RTF<1 才能不积压；
- 优化优先级：解码（大头）→ 前端 → 网络；
- 面试指标三件套：**TTFB、RTF、WER/UER**——报 P50 也要报 P95。

In [31]:
# 延迟分解条形图：主战场=解码
comp = {'前端(降噪/VAD)': 40, '特征提取': 15, 'ASR 解码': 180, '后处理': 25, '网络/调度': 30}
plt.barh(list(comp), list(comp.values()), color='C3')
plt.xlabel('延迟 (ms)'); plt.title('端到端延迟分解（P50）')
plt.tight_layout(); plt.show()
for k, v in comp.items(): print('%s: %d ms' % (k, v))
print('总延迟 = %d ms → 优化先砍解码（剪枝/量化/早停）' % sum(comp.values()))

C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3381845710.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


前端(降噪/VAD): 40 ms
特征提取: 15 ms
ASR 解码: 180 ms
后处理: 25 ms
网络/调度: 30 ms
总延迟 = 290 ms → 优化先砍解码（剪枝/量化/早停）


In [32]:
# 识别混淆矩阵：定位“谁和谁常混”
labels = ['0', '1', '2', '3']
conf = np.array([[0.93, 0.04, 0.02, 0.01],
                 [0.03, 0.90, 0.05, 0.02],
                 [0.02, 0.06, 0.88, 0.04],
                 [0.01, 0.03, 0.05, 0.91]])
plt.imshow(conf, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(); plt.xticks(range(4), labels); plt.yticks(range(4), labels)
for i in range(4):
    for j in range(4):
        plt.text(j, i, '%.2f' % conf[i, j], ha='center', va='center', fontsize=9)
plt.xlabel('识别结果'); plt.ylabel('真实标签')
plt.title('混淆矩阵：2↔3 混淆最高 → 近似音对，加音系特征/热词')
plt.tight_layout(); plt.show()
print('整体准确率 = %.1f%%' % (100*conf.diagonal().mean()))

C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3414376687.py:14: UserWarning: Glyph 8596 (\N{LEFT RIGHT ARROW}) missing from font(s) Microsoft YaHei.
  plt.tight_layout(); plt.show()


整体准确率 = 90.5%


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3414376687.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# WER-SNR 曲线（模拟）：噪声越强性能越差（测试必报条件）
snr = np.linspace(-5, 25, 13)
wer = 0.02 + 0.35*np.exp(-(snr+5)/6) + 0.01*snr/25
plt.plot(snr, wer, 'o-')
plt.xlabel('SNR (dB)'); plt.ylabel('WER')
plt.title('WER-SNR 包络：远场+噪声是真实挑战（15-25dB 才是常见场景）')
plt.tight_layout(); plt.show()
print('20dB → WER %.2f | 5dB → WER %.2f（差一倍多 → 前端降噪的价值）' % (wer[9], wer[3]))

20dB → WER 0.04 | 5dB → WER 0.12（差一倍多 → 前端降噪的价值）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\4099307319.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧩 链路案例题：一句话模板

**「线上 WER 突然涨 30%，怎么查？」**

1. 先验数据：时段/设备/地区分布 → 定向（某渠道新机型/录音链路变更）；
2. 查前端：AEC/VAD 配置、回声残留、采样率变化；
3. 查特征/模型：CMVN 漂移、量化位宽、上线模型 hash；
4. 查解码/后处理：热词表、LM 版本、标点规则；
5. 回滚 + 复盘：把指标拆成「前端/声学/语言/工程」四因子归因。

答题顺序 = 排查顺序：**先数据后假设，先分层后动刀**。

## 🏁 语音终面案例 5 连

1. 「设计车载语音交互」→ 阵列+回声+降噪+打断（barge-in）；
2. 「多语种识别怎么做」→ 共享编码器+lang 前缀/多任务；
3. 「语音 LLM 的幻觉怎么控」→ 上下文约束+检索+置信拒识；
4. 「TTS 情感可控怎么落地」→ 风格 token/情感标签+F0 控制；
5. 「声纹防伪（ASVspoof）」→ 合成检测器+信道一致性+活体信号。

每题按「需求→指标→架构→数据→风险→排期」六段式作答，语音方向即通关。

In [34]:
# 声纹识别流程框图：注册 → 向量化 → 比对 → 判决
fig, ax = plt.subplots(figsize=(9.5, 3.2))
n1 = [('注册语音', 0.02, '#ffe9d9'), ('特征/嵌入\n(说话人向量)', 0.25, '#dbe9ff'), ('向量库\n(每说话人 1~N)', 0.48, '#e2efda')]
n2 = [('测试语音', 0.02, '#ffe9d9'), ('同一嵌入网络', 0.25, '#dbe9ff'), ('相似度 cos', 0.48, '#e2efda'), ('阈值判决', 0.71, '#b45309')]
def draw(nodes, yoff):
    for label, x, fc in nodes:
        ax.add_patch(plt.Rectangle((x, yoff), 0.2, 0.28, fc=fc, ec='#1f4e79', lw=1.1))
        ax.text(x+0.1, yoff+0.14, label, ha='center', va='center', fontsize=8)
    for i in range(len(nodes)-1):
        x1 = nodes[i][1]+0.2; x2 = nodes[i+1][1]
        ax.annotate('', xy=(x2-0.002, yoff+0.14), xytext=(x1+0.002, yoff+0.14),
                    arrowprops=dict(arrowstyle='->', lw=1.1))
draw(n1, 0.58); draw(n2, 0.14)
ax.plot([0.45, 0.45], [0.28, 0.56], color='#555', lw=1, ls='--')
ax.annotate('', xy=(0.45, 0.56), xytext=(0.45, 0.58), arrowprops=dict(arrowstyle='->', lw=1, color='#555'))
ax.text(0.5, 0.015, '声纹 vs 语音识别：识别“谁在说” vs “说了什么”——共享特征，不同目标函数', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('说话人验证/识别：注册-比对两阶段（cos 相似度 + 阈值）', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 训练目标：同人向量近、异人向量远（三元组/GE2E 损失）')

✓ 训练目标：同人向量近、异人向量远（三元组/GE2E 损失）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\1053501517.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [35]:
# 流式 vs 非流式 ASR 对比框图
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.text(0.15, 0.80, '非流式（离线）', ha='center', fontsize=9, color='#1f4e79')
ax.add_patch(plt.Rectangle((0.03, 0.55), 0.24, 0.2, fc='#dbe9ff', ec='#1f4e79', lw=1.1))
ax.add_patch(plt.Rectangle((0.35, 0.55), 0.24, 0.2, fc='#dbe9ff', ec='#1f4e79', lw=1.1))
ax.text(0.15, 0.65, '完整语音', ha='center', fontsize=8)
ax.text(0.47, 0.65, '全部帧一起解码\n(整句上下文)', ha='center', fontsize=8)
ax.annotate('', xy=(0.348, 0.65), xytext=(0.274, 0.65), arrowprops=dict(arrowstyle='->', lw=1.1))
ax.text(0.15, 0.16, '流式（在线）', ha='center', fontsize=9, color='#376027')
for i, lab in enumerate(['语音分块', '增量解码\n(部分结果)', '流式输出']):
    x = 0.03 + i*0.3
    ax.add_patch(plt.Rectangle((x, 0.02), 0.24, 0.2, fc='#e2efda', ec='#376027', lw=1.1))
    ax.text(x+0.12, 0.12, lab, ha='center', fontsize=8)
    if i < 2:
        ax.annotate('', xy=(x+0.298, 0.12), xytext=(x+0.244, 0.12), arrowprops=dict(arrowstyle='->', lw=1.1, color='#376027'))
ax.text(0.5, 0.01, '', fontsize=1)
ax.text(0.98, 0.08, '边说边出：延迟低，但上下文受限 → 准确率略降', ha='right', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('流式(上=离线/下=在线)：上下文长度 vs 延迟的工程权衡', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 流式三要素：分块输入、增量解码、可修正输出（如 partial/final 事件）')

✓ 流式三要素：分块输入、增量解码、可修正输出（如 partial/final 事件）


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\1025515993.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 工程指标公式（默写版）

**SNR**：$\text{SNR}=10\log_{10}\dfrac{P_{\text{信号}}}{P_{\text{噪声}}}$（dB，>20 算清晰，<10 算嘈杂）。

**RTF**：$\text{RTF}=\dfrac{T_{\text{处理}}}{T_{\text{音频}}}$；$\text{RTF}<1$ 即实时；$\text{实时率}=1/\text{RTF}$（1 核能跑几倍速）。

**首包延迟（E2E 延迟）**：$L=L_{\text{前端}}+L_{\text{特征}}+L_{\text{解码}}+L_{\text{网络}}$（P50 取中位数，P95 看尾延迟）。

**WER**：$\text{WER}=\dfrac{\text{Sub}+\text{Ins}+\text{Del}}{N}$（编辑距离 / 参考词数；中文常用 CER）。

**LMS 回声消除**：$e=d-\mathbf{w}^\top\mathbf{x}$；$\mathbf{w}\leftarrow\mathbf{w}+2\mu\,e\,\mathbf{x}$（$\mu$ 步长，收敛条件 $0<\mu<1/\lambda_{\max}$）。

**波束形成（delay-and-sum）**：$y=\sum_{m=1}^{M} x_m(t-\tau_m)$，$\tau_m$ 对齐到声源方向 → M 路 SNR 提升约 $10\log_{10}M$ dB。

> 面试必背：RTF<1 才算实时；WER 用编辑距离；回声=远端参考信号可估计。

In [36]:
# SNR 数字示例：已知信号/噪声能量 → 分贝
s = 0.1*np.sin(2*np.pi*300*np.arange(800)/16000)
n = 0.01*np.random.RandomState(2).randn(800)
x = s + n
snr = 10*np.log10(np.sum(s**2)/np.sum(n**2))
print('信号能量 = %.4f | 噪声能量 = %.4f | SNR = %.2f dB' % (np.sum(s**2), np.sum(n**2), snr))
assert 16 < snr < 24
print('✓ 0.1 振幅正弦 vs 0.01 高斯噪声 ≈ 17dB：每 +10dB 噪声能量降 10 倍')

信号能量 = 4.0000 | 噪声能量 = 0.0827 | SNR = 16.85 dB
✓ 0.1 振幅正弦 vs 0.01 高斯噪声 ≈ 17dB：每 +10dB 噪声能量降 10 倍


In [37]:
# RTF 数字示例：实时率与并行吞吐
rtf = 0.25                      # 处理 1s 音频耗时 0.25s
print('RTF = %.2f → 单核实时率 = %.1f×（可同时处理音频的 N 倍速）' % (rtf, 1/rtf))
print('16 核并行 → 音频总吞吐 ≈ %.0f×实时（远高于 1× 才敢上生产）' % (16/rtf))
assert abs(1/rtf - 4.0) < 1e-9
print('✓ 指标链：RTF → 实时率 → 核心数 → 并发路数（面试喜欢问这串换算）')

RTF = 0.25 → 单核实时率 = 4.0×（可同时处理音频的 N 倍速）
16 核并行 → 音频总吞吐 ≈ 64×实时（远高于 1× 才敢上生产）
✓ 指标链：RTF → 实时率 → 核心数 → 并发路数（面试喜欢问这串换算）


## 📜 工程算法伪代码（30 行级）

**LMS 回声消除**：
```
for n in 0..N:
    y = w @ x[n:n+M]              # 远端信号估计的回声
    e = d[n] - y                  # 近端 - 回声估计 = 残余
    w += 2*mu * e * x[n:n+M]      # 更新滤波器
```

**VAD（能量+过零率双判）**：
```
E_t = mean(x²); Z_t = |sign(diff(x))| 平均
语音帧 = E_t > θE 且 Z_t < θZ（浊音过零率低）
```

**WER 计算**：DP 编辑距离 → `(sub+ins+del)/N_ref`。

> 加餐：热词提升 = 解码图动态加权重；降噪 = 频谱减法/深度学习掩码。

## 🎤 工程追问（8 连·第二轮）

Q「RTF=0.25 能上多路？」→ 1/0.25=4 路/核，16 核≈64 路（不计资源竞争）；
Q「回声消除参考信号哪来？」→ 远端播放音频缓冲（speaker 输出）——无参考就只能降噪；
Q「VAD 误检静音会怎样？」→ 前半段被吞 → 首词丢失 → 流式场景损失更大；
Q「热词怎么加？」→ 解码前缀加权/偏置语言模型；
Q「说话人向量用什么损失？」→ 三元组/GE2E/软代理：同人内聚、异人分离；
Q「波束形成要几个麦？」→ ≥2；线性阵>4 提升明显，圆形阵适合会议；
Q「降噪会不会伤识别？」→ 过度降噪丢语音细节 → 用掩码+保留底噪的折中；
Q「首包延迟怎么优化？」→ 前端降延迟 + 流式解码 + 分块声码器/特征 + 网络长连接。

## ✅ 自测（10 题）

1. SNR 20dB 意味着什么？（信号功率是噪声 100 倍）
2. RTF 定义与实时条件？（处理/音频时长，<1）
3. WER 三错误类型？（替换/插入/删除）
4. 回声消除需要什么额外输入？（远端参考）
5. 流式识别为什么准确率略降？（上下文受限）
6. 声纹识别用什么相似度？（cos，配阈值）
7. 降噪的两类方法？（传统谱减 / 深度学习掩码）
8. 热词的作用场景？（专名/产品词纠错）
9. 多麦克风为什么更好？（空间选择性 + SNR 提升）
10. 端到端与传统在工程上的取舍？（灵活/准确 vs 可解释/成熟）

In [38]:
# WER 数字示例：编辑距离 DP 手算
def lev(a, b):
    d = np.zeros((len(a)+1, len(b)+1), dtype=int)
    for i in range(len(a)+1): d[i, 0] = i
    for j in range(len(b)+1): d[0, j] = j
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            d[i, j] = min(d[i-1, j]+1, d[i, j-1]+1,
                         d[i-1, j-1] + (a[i-1] != b[j-1]))
    return d[-1, -1]
ref = ['我', '爱', '北', '京']; hyp = ['我', '爱', '北', '大']
ed = lev(ref, hyp)
wer = ed/len(ref)
print('参考 4 字 vs 识别 4 字 → 编辑距离 = %d → WER = %.0f%%' % (ed, 100*wer))
assert ed == 1 and abs(wer - 0.25) < 1e-9
print('✓ 1 处替换：换字/增字/漏字各计 1 → 分母=参考长度（中文常用 CER）')

参考 4 字 vs 识别 4 字 → 编辑距离 = 1 → WER = 25%
✓ 1 处替换：换字/增字/漏字各计 1 → 分母=参考长度（中文常用 CER）


In [39]:
# 传统 vs 端到端雷达图：六维对比
labels = ['准确率', '鲁棒性', '灵活性', '可解释性', '实时性', '生态成熟度']
trad = [88, 92, 55, 85, 85, 90]
e2e = [96, 84, 92, 55, 78, 88]
ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]
fig, ax = plt.subplots(figsize=(5.8, 5.0), subplot_kw=dict(polar=True))
for name, v in [('传统 (HMM/WFST)', trad), ('端到端 (CTC/AED)', e2e)]:
    ax.plot(ang, v + v[:1], 'o-', lw=1.6, label=name)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=9)
ax.set_yticks([50, 75, 100]); ax.set_ylim(40, 105)
ax.legend(loc='lower right', fontsize=9); ax.set_title('两代架构：互有取舍，按场景选型', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 面试金句：没有银弹——低资源/嵌入式选传统，高精度/快速迭代选端到端')

✓ 面试金句：没有银弹——低资源/嵌入式选传统，高精度/快速迭代选端到端


C:\Users\24260\AppData\Local\Temp\ipykernel_34508\3630476709.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话开篇：一个语音产品里藏着哪些“小算法”？

你以为识别只是“模型吃音频吐文字”？真实产品里，音频先过一串“预处理守卫”：

```
麦克风 → VAD(有人说话吗) → AEC(去掉自己播的声) → 降噪 → 波束(多麦选方向)
→ 说话人日志(谁在说) → ASR → NLU → 应答
```

**VAD（语音活动检测）**：判断“这段有没有人声”——没人声就不喂 ASR，省 90% 算力与误识别。
经典信号：能量（说=大声）、过零率（说话=高频成分多）。

**AEC（回声消除）**：音箱放音乐 + 你说“小爱同学” → 麦克风两样都收。
思路：**我早知道播放了什么**（参考信号）→ 用自适应滤波估计“播放声传到麦克风的路径”
→ 从麦克风里减掉它 → 只剩你的声音（本书 LMS 示例）。

**波束形成**：多麦 = 多个耳朵 → 通过延时相加，让“目标方向的声音”叠加变强、
“其它方向”互相抵消 → 空间选择性（“指向性”）。

**一句话**：ASR 是主角，VAD/AEC/波束是让主角“只听到该听的”的保镖团。

## 🍵 白话讲透回声消除：学会了“自己声音的副本”再减掉

**问题**：设备在播放（音乐），你说“暂停”。麦克风收到的 = 你的话 + 音乐的**回声版**。
怎么只留你的话？——**回声是播放信号的“变形”**（过了扬声器→空气→麦克风的路径）。

**LMS 自适应滤波（三步）**：
1. 拿播放信号 x 的若干历史帧，加权组合 → 预测“此刻麦克风里的回声部分”；
2. 用麦克风真实值 y 减掉预测 → 残差 e（≈你的话 + 没消干净的）；
3. 按残差调整权重：**e 大 → 权重大步改；e 小 → 小步改**（梯度下降）。

**比喻**：第一次听伴奏猜歌词（权重瞎猜），唱一遍发现“我唱的和原唱差好多”
（e 大）→ 大幅度改；唱几遍差得少了 → 微调。最后权重 ≈ 真实回声路径 → 减得干净。

**为什么难**：路径随时间变（人走动、音量变）→ 要**在线**一直学；
回声有延迟（音箱到麦有几十 ms）→ 滤波器要多记忆几百个历史样本（抽头多）。

**一句话**：AEC = 在线学“播放→麦克风”的通道，然后把它的输出从麦克风里“对消”掉。

## 🍵 白话讲透声纹：每个人的“声音指纹”

**声纹识别**（说话人识别）回答两件事：
- **确认**：你说“我是张三”，对不对？（1:1 比对，门禁场景）；
- **辨识**：这段是谁说的？（1:N 搜索，会议分人）。

**原理**：训练一个网络把“任意长度的语音”压成**一个固定维度向量**（说话人向量/embedding），
向量越近 = 越像同一个人的声音。

**怎么训练（经典 GE2E/三元组思路）**：
- 同一人不同句子 → 向量要聚在一起；
- 不同人 → 向量要拉开；
- 损失 = 距离/相似度函数，端到端学“什么样的特征最分人”。

**为什么不是“音高”就行**：音高（基频）很容易模仿；声纹用的是**声道形状**决定的长期谱包络——
很难伪装。但也防不了“录音重放”→ 工程上加活体检测（问随机问题）。

**一句话**：声纹 = 把语音映射到“说话人空间”的一个点，靠向量距离判人。

## 🔍 公式拆解 ①：LMS 自适应滤波

$$y_t=w^{\top}x_t,\qquad e_t=d_t-y_t,\qquad w\leftarrow w+2\mu\,e_t\,x_t$$

**逐符号解释**：
- $x_t$：参考信号最近 L 个样本组成的向量（播放内容）；$w$：自适应权重（要学的东西）；
- $y_t=w^{\top}x_t$：模型预测的“回声副本”；$d_t$：麦克风实际值（回声+目标）；
- $e_t=d_t-y_t$：残差——**越接近 0 说明消得越干净**；
- $\mu$：步长（学习率）——决定“改多大”；
- 更新：残差大 → 权重大步改；残差小 → 小步改；$x_t$ 指向“该往哪改”。

**为什么这样改就能收敛（直觉）**：$e^2$ 是权重的二次函数（碗形），
$2e_t x_t$ 正是 $\nabla_w e_t^2$ → 每次沿负梯度走一小步 → 掉进碗底（最优权重）。

**μ 怎么选（面试必问）**：收敛要求 $0<\mu<1/\lambda_{\max}(R_{xx})$（相关阵最大特征值），
工程经验：μ 太大震荡不收敛、太小收敛慢——固定值不够时用**归一化 LMS**（NLMS：按 $\|x_t\|^2$ 缩放）。

**复杂度**：每次 O(L)——L=512 抽头也毫秒级，适合实时。

## 🔍 公式拆解 ②：波束形成的两板斧

**延时求和（Delay-and-Sum，DS）**：
$$y_t=\frac{1}{M}\sum_{m=1}^{M}x_m\Big(t-\frac{d_m\sin\theta}{c}f_s\Big)$$

**逐符号解释**：
- $M$：麦克风数；$x_m(t)$：第 m 个麦的采样；$d_m$：该麦到参考点的距离；
- $\theta$：目标声源方向角；$c$：声速（≈340 m/s）；$f_s$：采样率；
- $\frac{d\sin\theta}{c}f_s$：声波到各麦的**到达时间差**（以采样点计）；
- 操作：先“对齐”（把各麦信号按 TDOA 平移），再平均——
目标方向**同相相加**（×M 倍功率），其它方向**乱相抵消**。

**MVDR（最小方差无失真响应）**：
$$w=\frac{R_{nn}^{-1}\,a}{a^{H}R_{nn}^{-1}a},\qquad y=w^{H}x$$

- $a$：导向矢量（目标方向的相位差模式）；$R_{nn}$：噪声协方差矩阵；
- $R_{nn}^{-1}$：先“白化”噪声（压掉噪声方向），再投影到目标方向；
- 约束：目标方向增益=1（无失真），噪声方向方差最小。

**一句话**：DS=对齐+平均（简单鲁棒）；MVDR=白化+投影（噪声相关性大时更优）。

## 🔍 公式拆解 ③：WER 与编辑距离 DP

$$D(i,j)=\min\begin{cases}D(i-1,j)+1 &\text{删除 ref 的字}\\D(i,j-1)+1 &\text{插入 hyp 的字}\\D(i-1,j-1)+\delta &\text{替换（δ=0 若相同，否则 1）}\end{cases}$$

**逐符号解释**：
- ref：正确答案的词序列；hyp：识别结果；$D(i,j)$：ref 前 i 词 vs hyp 前 j 词的最小编辑距离；
- 三种基本操作：删（ref 多一个词）、插（hyp 多一个词）、替换（词错了）；
- $D(0,0)=0$；边界 $D(i,0)=i$、$D(0,j)=j$；
- 终点 $D(|ref|,|hyp|)$ = 最少操作数。

**WER 公式**：
$$\text{WER}=\frac{S+D+I}{N}=\frac{\text{替换}+\text{删除}+\text{插入}}{\text{参考词数}}$$

- 分子三类错误加起来，分母用 **ref 的词数**（不是 hyp）——所以 WER 可能 >100%；
- 用**最小编辑距离对齐**得到 S/D/I（同一对替换不能又算删又算插）；
- **直觉**：WER = “识别文本离正确文本差多少词”，越小越好。

In [40]:
# NLMS 回声消除数字示例：学习“回声路径”并逐步对消
rng = np.random.RandomState(1)
L = 32                                        # 滤波器抽头
h_true = 0.6*np.exp(-np.arange(L)/6.0)        # 真实回声路径（指数衰减）
N = 4000
x = rng.randn(N)                              # 播放信号（近白噪声）
echo = np.convolve(x, h_true)[:N]
near = 0.4*rng.randn(N)                       # 近端说话（你的声音）
mic = echo + near                             # 麦克风收到的一切
e0 = np.abs(mic[L:3*L]).mean()                # 未学习时的残差（= 回声+近端）
w = np.zeros(L); mu = 0.3                     # NLMS：按 ||x||² 归一化步长
err_curve = []
for t in range(L, N):
    xt = x[t-L+1:t+1][::-1]
    e = mic[t] - w @ xt
    w += 2*mu*e*xt/(xt @ xt + 1e-6)
    if t % 40 == 0: err_curve.append(abs(e))
e1 = np.mean(err_curve[-20:])
print('学习前残差 = %.3f → 学习后 = %.3f（消掉 %.0f%%）' % (e0, e1, 100*(1-e1/e0)))
print('学到的路径 w[:5] =', np.round(w[:5], 3), '| 真实路径 h[:5] =', np.round(h_true[:5], 3))
assert e1 < e0*0.6
print('✓ 残差 = 你声音+未消回声；NLMS 步长按信号能量归一 → 稳且快（工程标配）')

学习前残差 = 0.781 → 学习后 = 0.409（消掉 48%）


学到的路径 w[:5] = [0.583 0.544 0.33  0.314 0.341] | 真实路径 h[:5] = [0.6   0.508 0.43  0.364 0.308]
✓ 残差 = 你声音+未消回声；NLMS 步长按信号能量归一 → 稳且快（工程标配）


In [41]:
# 双麦延时求和：对准目标方向 → SNR 提升
fs = 16000; c = 340.0; d = 0.05                    # 麦间距 5cm
theta = 30.0                                       # 声源在 30°
tau = d*np.sin(np.deg2rad(theta))/c*fs             # 到达差（采样点）
t = np.arange(4000)/fs
s = np.sin(2*np.pi*500*t) + 0.4*np.sin(2*np.pi*1200*t)   # 目标语音
rng = np.random.RandomState(2)
n1 = 1.0*rng.randn(4000); n2 = 0.9*rng.randn(4000)       # 两麦独立噪声
m1 = s + n1
m2 = np.roll(s, int(round(tau))) + n2                     # 第二麦晚到 tau 点
def snr(y, ref): return 10*np.log10(np.var(ref)/np.var(y-ref))
ys = 0.5*(m1 + np.roll(m2, -int(round(tau))))             # 对齐后平均
print('单麦 SNR = %.1f dB | 双麦 DS 后 = %.1f dB（提升 %.1f dB）'
      % (snr(m1, s), snr(ys, s), snr(ys, s) - snr(m1, s)))
assert snr(ys, s) > snr(m1, s)
print('✓ 同相相加目标声、异相抵消噪声 → 方向性增益 ≈ +3dB（M=2 理论值）')

单麦 SNR = -2.5 dB | 双麦 DS 后 = 1.0 dB（提升 3.5 dB）
✓ 同相相加目标声、异相抵消噪声 → 方向性增益 ≈ +3dB（M=2 理论值）


In [42]:
# WER 手算：最小编辑距离 DP 表 + 对齐可视化
ref = ['我', '爱', '北', '京']
hyp = ['我', '不', '爱', '京']
D = np.zeros((len(ref)+1, len(hyp)+1), dtype=int)
for i in range(len(ref)+1): D[i, 0] = i
for j in range(len(hyp)+1): D[0, j] = j
for i in range(1, len(ref)+1):
    for j in range(1, len(hyp)+1):
        D[i, j] = min(D[i-1, j]+1, D[i, j-1]+1, D[i-1, j-1] + (0 if ref[i-1] == hyp[j-1] else 1))
print('DP 表（行=ref，列=hyp）：\n', D)
ed = D[len(ref), len(hyp)]
wer = ed/len(ref)                                  # WER = 编辑距离 / 参考词数
print('编辑距离 = %d（最少操作数）| WER = %.0f%%（= 距离/参考词数）' % (ed, 100*ed/len(ref)))
assert ed == 2 and wer == 0.5
print('✓ 例子：我→我不(插/替换)，爱保留，北京→京(删+替) —— 共 2 次编辑，WER=50%')

DP 表（行=ref，列=hyp）：
 [[0 1 2 3 4]
 [1 0 1 2 3]
 [2 1 1 1 2]
 [3 2 2 2 2]
 [4 3 3 3 2]]
编辑距离 = 2（最少操作数）| WER = 50%（= 距离/参考词数）
✓ 例子：我→我不(插/替换)，爱保留，北京→京(删+替) —— 共 2 次编辑，WER=50%


## ⚠️ 本章高频踩坑（面试向）

1. **μ 步长**：LMS 步长过大直接发散；公式背 $0<\mu<1/\lambda_{\max}$；
2. **VAD 阈值**：能量阈值定死 → 噪声环境全误报/漏报；要自适应底噪（滑动统计）；
3. **AEC 双讲（double-talk）**：你说话时也在更新滤波器 → 把你的声音当回声消掉；要双讲检测暂停更新；
4. **回声路径时变**：人走动路径变 → 在线追踪（步长别太小）；
5. **WER 分母**：用 ref 词数（hyp 更长可能 >100%）；别和 CER 混（字 vs 词）；
6. **声纹防伪**：重放攻击必须靠活体检测；相似度阈值调错 = 误拒/误纳权衡（EER）；
7. **波束假设**：远场平面波 + 固定几何；近场/混响强会失效；
8. **数据泄露**：声纹/ASR 评测要分说话人（同人不同句进同集合 = 作弊）。

## 🧭 语音工程全景：模块、指标、选型

| 模块 | 输入→输出 | 核心指标 | 常用方案 |
|---|---|---|---|
| VAD | 音频→语音段 | 漏报/误报率 | 能量/过零/神经网络 |
| AEC | 麦+参考→净音 | ERLE、双讲表现 | LMS/NLMS/频域块 |
| 降噪 | 噪声音→净音 | SNR 提升、语音失真 | 谱减法/RNN-NM |
| 波束 | 多麦→单路 | 方向增益、DOA 误差 | DS/MVDR/GSC |
| 声纹 | 语音→向量 | EER、DCF | x-vector/ECAPA |
| ASR | 音频→文本 | WER/CER | CTC/AED/RNN-T |

**选型金句**：
- 先问**部署形态**（实时/离线/端侧/云端）再选方案；
- 传统信号处理（VAD/AEC/波束）在工程里**不会消失**——它们便宜、可控、离线友好；
- 大模型语音（Whisper/GPT-4o-voice）能替代部分模块，但**时延/算力/幻觉**仍是门槛。

## 🏁 本章学完清单

- [ ] 能画出语音产品流水线并说明每模块管什么；
- [ ] 能用比喻讲清 AEC（学回声副本再减掉）；
- [ ] 能手写 LMS 三公式并解释 μ 的取值约束；
- [ ] 能解释 DS 为什么有方向性、MVDR 为什么更好；
- [ ] 能手算编辑距离 DP 表（本书示例）并写 WER 公式；
- [ ] 能讲清声纹确认 vs 辨识、EER 概念；
- [ ] 能说出 3 个工程坑（双讲/步长/重放攻击）；
- [ ] 能对比各模块方案并给出选型逻辑。

> 语音四本全部学完 —— 现在去数据科学侧继续（下一目录）。

## 🗺️ 16-语音 完整知识体系地图（9 本）

```
00 信号基础与特征     ← 一切的地基（STFT/窗/MFCC/CMVN）
01 GMM-HMM/DTW        ← 经典识别（先验概率/维特比/对齐）
02 CTC/Attention       ← 端到端 ASR（对齐/注意力/梯度）
03 语音合成TTS         ← Tacotron2/VITS/声码器
04 语音工程八股        ← 全链路工程（VAD/AEC/波束/WER）← 你在这
05 语音增强与降噪      ← 谱减/维纳/掩码/神经增强/PESQ-STOI-SI-SDR
06 声纹与说话人日志    ← x-vector/ECAPA/EER/PLDA/AHC
07 语音转换与歌声转换  ← VQ-VAE/AutoVC/StarGAN/HuBERT/So-VITS-SVC
08 语音大模型与智能体  ← Whisper/音频token(RVQ)/RNN-T/端到端对话
```

- 已读 00-04 → 建议 05（前端增强）→ 06/07（身份与音色）→ 08（大模型收尾）；
- 面试主线：00→02 识别、03→08 合成/大模型、04 工程、06/07 身份与转换。

> 语音九本全部学完 —— 现在去数据科学侧继续（下一目录）。